# Class 30: Time Series Fitting: Exoplanet Transits
## Objective: Understand time series data for exoplanets

These exercises are based on those in "Lecture 19: Time Series Fitting: Exoplanet Transits" by Yuan-Sen Ting and available from https://tingyuansen.github.io/coding_essential_for_astronomers/lectures/lecture19-time-series-fitting-exoplanet-transits.html

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from astropy import units as u
from astropy import constants as const

# Configure matplotlib for better-looking plots
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10
# Plotting defaults
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Define solar and planetary constants using astropy
M_sun = const.M_sun.value  # kg
R_sun = const.R_sun.value  # m
G = const.G.value  # m^3 kg^-1 s^-2
M_earth = const.M_earth.value  # kg
R_earth = const.R_earth.value  # m
M_jupiter = const.M_jup.value  # kg
R_jupiter = const.R_jup.value  # m

# Set random seed for reproducibility
np.random.seed(42)

## Section 1: Exoplanet radial velocity variations

The radial velocity precision needed to detect an exoplanet is extremely high. 

The simple reason why is that we find evidence for exoplanets through the motion of their star in response to the planet's gravity. Stars and planets orbit around their center of mass, and planets are **much** less massive than stars. For example, Jupiter is 1000 times less massive than the Sun, and the Earth is over 300 times less massive than Jupiter. 

The first exoplanets were detected in the 1990s with spectrographs whose radial velocity precision was about 100 m/s. Unsurprisingly, they were only able to detect the most massive planets in relatively short orbits (tens of days) around their parent stars

Over the last few decades, radial velocity precision has improved so that now the best instruments have obtained close to 0.1 m/s precision. This is approximately comparable to the motion of the Sun in response to the Earth at one AU. 

In addition to radial velocity precision, the positive detection of an exoplanet requires measurements for roughly a complete orbit. The first exoplanet discoveries favored short-period orbits as it did not require as long to measure a complete orbit. For an Earth-like planet around a Sun-like star, observations need to span approximately a year. 

In [ ]:
# System parameters (Earth-Sun, edge-on orbit)
m_star = 1.0 * M_sun  # Star mass (Sun)
m_planet = 1.0 * M_earth  # Planet mass (Earth)
period = (365.25 * u.day).to(u.s).value  # Orbital period in seconds
sin_i = 1.0  # Assume edge-on orbit (maximum signal)

# Calculate RV amplitude directly from the formula
# K ≈ (2πG/P)^(1/3) × (M_p sin i) / M_star^(2/3)
K = (2 * np.pi * G / period)**(1/3) * (m_planet * sin_i) / (m_star)**(2/3)

# Convert to readable units
K_ms = (K * u.m / u.s).to(u.m / u.s).value
K_cms = (K * u.m / u.s).to(u.cm / u.s).value
print(f"Earth's RV amplitude: K = {K_ms:.4f} m/s = {K_cms:.2f} cm/s")

# Create a comparison table of different planets
planets = [
    ("Hot Jupiter (3d)", M_jupiter, 3.0),
    ("Warm Neptune (20d)", 17*M_earth, 20.0),
    ("Super-Earth (50d)", 5*M_earth, 50.0),
    ("Earth twin (365d)", M_earth, 365.25),
]

print("\nRadial Velocity Amplitudes for Different Planets:")
print("-" * 60)
print(f"{'Planet Type':<20} {'Mass':<15} {'Period':<10} {'K (m/s)':<10}")
print("-" * 60)

for name, mass, period_days in planets:
    period_s = (period_days * u.day).to(u.s).value
    
    # Calculate RV amplitude
    K = (2 * np.pi * G / period_s)**(1/3) * (mass * sin_i) / (m_star)**(2/3)
    
    # Mass in Earth masses for display
    mass_earth = mass / M_earth
    
    print(f"{name:<20} {mass_earth:>6.1f} M⊕  {period_days:>8.1f}d  {K:>8.2f}")

**Test your understanding:** 
Add the planet Jupiter to the list of planets and update the table. What is the radial velocity amplitude of Jupiter? Note Jupiter orbits the Sun once every 11.86 Earth years.

## Section 2: Exoplanet transit flux variations

Transit detections require extraordinarily good photometry. The depth of a transit is proportional to the area of the planet to the area of the star that it blocks. 

$$
\mathrm{Transit\,Depth} = \frac{ \mathrm{Area\,of\,Planet}}{\mathrm{Area\,of\,Star}} = \frac{\pi R_p^2}{\pi R_*^2} \left(\frac{R_p}{R_*} \right)^2
$$


The transit depths are typically so small that they are commonly expressed in parts per million (ppm). An Earth-like planet around a Sun-like star produces an 84 ppm transit depth, which is so small that it is essentially impossible to measure from the ground due to atmospheric turbulence. 

Space missions like Kepler and TESS have achieved extraordinary sensitivity (tens of ppm) in observations of various lengths. 

Here are some transit depths: 

In [ ]:
# Calculate transit depths
depth_earth = (R_earth / R_sun)**2
depth_jupiter = (R_jupiter / R_sun)**2

print("Transit Depths for Planets Transiting the Sun:")
print("-" * 50)
print(f"Earth:")
print(f"  Radius ratio: {R_earth/R_sun:.5f}")
print(f"  Transit depth: {depth_earth*1e6:.1f} ppm = {depth_earth*100:.5f}%")

print(f"\nJupiter:")
print(f"  Radius ratio: {R_jupiter/R_sun:.4f}")
print(f"  Transit depth: {depth_jupiter*1e6:.0f} ppm = {depth_jupiter*100:.2f}%")

## Section 3: Transit Probability

Transits require that the orbit be nearly edge on from our perspective, as otherwise the planet will never pass between us and the star. 

The probability of a transit depends on both how close the planet orbits the star and the size of the planet. The probability is 

$$
P_{transit} \approx \frac{R_*}{a}
$$
The critical angle is $\cos i \approx R_*/a$, the maximum tilt angle at which the planet still at least slightly transits from our perspective. Here are some calculations of the transit probability:

In [ ]:
# Calculate transit probabilities for different orbital distances
# Using P_transit ≈ R_star / a
from astropy.constants import R_sun

# Define stellar radius (Sun's radius)
R_star = R_sun.to(u.AU)

# Define orbital distances
a_hot_jupiter = 0.05 * u.AU
a_earth = 1.0 * u.AU

# Calculate transit probabilities
P_transit_hot_jupiter = (R_star / a_hot_jupiter).decompose()
P_transit_earth = (R_star / a_earth).decompose()

print("Transit Probability Calculations")
print("=" * 50)
print(f"Stellar radius: R_sun = {R_star:.5f}")
print()
print(f"Hot Jupiter at {a_hot_jupiter}:")
print(f"  P_transit ≈ {R_star:.5f} / {a_hot_jupiter}")
print(f"  P_transit ≈ {P_transit_hot_jupiter:.4f} = {P_transit_hot_jupiter*100:.2f}%")
print()
print(f"Earth at {a_earth}:")
print(f"  P_transit ≈ {R_star:.5f} / {a_earth}")
print(f"  P_transit ≈ {P_transit_earth:.5f} = {P_transit_earth*100:.3f}%")
print()
print(f"Ratio: Hot Jupiter is {P_transit_hot_jupiter/P_transit_earth:.1f}× more likely to transit")

**Test your understanding:** What is the transit probability for Jupiter at 5.2 AU? Is it larger or smaller than the probability for the Earth? 

In [ ]:
# Enter your code here
a_jupiter = 5.2 * u.AU


## Section 4: Transit Geometry

The probability depends on the size of the planet, and so the inclination does not have to be exactly $i = 90^\circ$. Furthermore, in some cases the planet may only graze the disc of the star, rather than pass completely across it. This can give rise to different transit light curve shapes. 

The common quantity used to describe the alignment of the planet's orbit relative to our line of sight is the impact parameter $b = a \cos i/R_*$. 

In [ ]:
# Planet passing directly across the star

import matplotlib.patches as patches

# Create figure: 2 rows × 4 columns (top = geometry, bottom = light curve)
fig, axes = plt.subplots(2, 4, figsize=(14, 7))

# Define four key transit phases with planet positions
positions = [
    (-1.5, 0, "Before transit"),
    (-0.5, 0, "Ingress"),
    (0, 0, "Mid-transit"), 
    (0.5, 0, "Egress"),
]

# Set stellar and planetary radii for visualization
R_star = 1.0
R_planet = 0.15  # 15% of star's radius (exaggerated for visibility)

# Create time array and light curve
times = np.linspace(-2, 2, 200)
flux = np.ones_like(times)

# Simple box transit (flat-bottomed)
transit_start = -0.7
transit_end = 0.7
in_transit = (times > transit_start) & (times < transit_end)
flux[in_transit] = 1 - R_planet**2  # Depth = (Rp/Rs)²

# Loop through each transit phase and create the visualization
for i, (x_planet, y_planet, label) in enumerate(positions):
    ax_top = axes[0, i]
    ax_bottom = axes[1, i]
    
    # Top panel: Draw star and planet at this phase
    star = patches.Circle((0, 0), R_star, color='gold', ec='orange', lw=2)
    ax_top.add_patch(star)
    
    planet = patches.Circle((x_planet, y_planet), R_planet, color='black', alpha=0.8)
    ax_top.add_patch(planet)
    
    ax_top.set_xlim(-2, 2)
    ax_top.set_ylim(-1.5, 1.5)
    ax_top.set_aspect('equal')
    ax_top.axis('off')
    
    # Add arrow showing planet motion direction
    if i < 3:
        ax_top.arrow(x_planet + R_planet + 0.1, 0, 0.3, 0, 
                    head_width=0.1, head_length=0.1, fc='blue', ec='blue')
    
    # Bottom panel: Show light curve with current phase marked
    ax_bottom.plot(times, flux, 'k-', lw=2)
    
    # Determine current position on light curve for each phase
    if i == 0:  # Before transit
        current_time = -1.5
        current_flux = 1.0
    elif i == 1:  # Ingress (partial blocking)
        current_time = -0.5
        current_flux = 1 - 0.5 * R_planet**2
    elif i == 2:  # Mid-transit (maximum blocking)
        current_time = 0
        current_flux = 1 - R_planet**2
    else:  # Egress (partial blocking)
        current_time = 0.5
        current_flux = 1 - 0.5 * R_planet**2
    
    # Mark current position with red dot
    ax_bottom.plot(current_time, current_flux, 'ro', markersize=8)
    
    ax_bottom.set_xlim(-2, 2)
    ax_bottom.set_ylim(0.975, 1.005)
    ax_bottom.set_xlabel('Time', fontsize=13)
    if i == 0:
        ax_bottom.set_ylabel('Relative Flux', fontsize=13)
    ax_bottom.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Planet passing in front of the star at different inclination angles

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Three impact parameter cases to compare
impact_params = [0.0, 0.5, 0.9]
labels = ["Central (b=0.0)", "Mid (b=0.5)", "Grazing (b=0.9)"]
colors = ['darkblue', 'purple', 'darkred']

# Star and planet sizes for visualization
R_star_plot = 1.0
R_planet_plot = 0.1
times_transit = np.linspace(-0.15, 0.15, 200)  # days

# Loop through each impact parameter case and create visualizations
for idx, (b, label, color) in enumerate(zip(impact_params, labels, colors)):
    ax_geom = axes[0, idx]
    ax_lc = axes[1, idx]
    
    # Top panel: Show transit geometry with planet path
    circle = plt.Circle((0, 0), R_star_plot, color='gold', ec='orange', lw=2)
    ax_geom.add_patch(circle)
    
    # Draw planet's path (horizontal line at height b)
    path_y = b * R_star_plot
    path_x = np.linspace(-1.3, 1.3, 50)
    ax_geom.plot(path_x, np.full_like(path_x, path_y), 'k--', lw=1, alpha=0.3)
    
    # Show planet at multiple positions along its path
    n_positions = 7
    for i, frac in enumerate(np.linspace(-1, 1, n_positions)):
        x_pos = frac * 1.2 * R_star_plot
        y_pos = path_y
        dist_from_center = np.sqrt(x_pos**2 + y_pos**2)
        if dist_from_center < R_star_plot + R_planet_plot:
            alpha_val = 0.7 if abs(frac) < 0.5 else 0.3
            planet_circle = plt.Circle((x_pos, y_pos), R_planet_plot, 
                                      color='black', alpha=alpha_val)
            ax_geom.add_patch(planet_circle)
    
    ax_geom.set_xlim(-1.5, 1.5)
    ax_geom.set_ylim(-1.5, 1.5)
    ax_geom.set_aspect('equal')
    ax_geom.set_title(label, fontsize=16, fontweight='bold')
    ax_geom.axis('off')
    
    # Annotate impact parameter if non-zero
    if b > 0:
        ax_geom.plot([0, 0], [0, path_y], 'r-', lw=2)
        ax_geom.text(0.1, path_y/2, f'b = {b}', fontsize=12, color='red', 
                    fontweight='bold')
    
    # Bottom panel: Generate trapezoidal transit light curve
    flux = np.ones_like(times_transit)
    
    # Transit parameters depend on impact parameter
    T_total = 0.12  # Total transit duration in days
    
    # Use SAME formula as trapezoidal_transit_model fitting function
    # From geometry: t_ingress / T_total = R_p / R_star
    T_ingress = T_total * R_planet_plot / R_star_plot  # Ingress/egress duration
    
    # Depth decreases with impact parameter (grazing transits are shallower)
    max_depth = R_planet_plot**2
    if b < 1 - R_planet_plot:
        # Planet fully enters disk
        depth = max_depth
        has_flat_bottom = True
    else:
        # Grazing: planet doesn't fully enter, reduced depth
        depth = max_depth * (1 - b)**2 / R_planet_plot**2
        has_flat_bottom = False
    
    for j, t in enumerate(times_transit):
        abs_t = np.abs(t)
        
        if abs_t < T_total / 2:
            if has_flat_bottom:
                # Trapezoidal shape
                if abs_t <= (T_total / 2 - T_ingress):
                    # Flat bottom (center of transit)
                    flux[j] = 1 - depth
                else:
                    # Ingress or egress (linear slopes near edges)
                    # Distance from edge of transit
                    dist_from_edge = T_total / 2 - abs_t
                    fraction_depth = dist_from_edge / T_ingress
                    flux[j] = 1 - depth * max(0, min(1, fraction_depth))
            else:
                # V-shaped (grazing): maximum depth at center, linear slopes to edge
                fraction_depth = 1 - abs_t / (T_total / 2)
                flux[j] = 1 - depth * max(0, fraction_depth)
    
    ax_lc.plot(times_transit * 24, flux, color=color, lw=2.5)
    ax_lc.set_xlabel('Time [hours]', fontsize=13)
    if idx == 0:
        ax_lc.set_ylabel('Relative Flux', fontsize=13)
    ax_lc.grid(True, alpha=0.3)
    ax_lc.set_xlim(-3.6, 3.6)

plt.tight_layout()
plt.show()

## Section 5: Fitting a single transit

Exoplanet transits have a typical geometry that is somewhat trapezoidal. There is the total depth, which depends on the area of the planet relative to the star. There is the duration of the transit, which depends on the size of the star and the orbital speed of the planet, and there is the length of the ingress / egress, which depends on the size of the planet. 

### Total Duration

The total duration of the transit is:

$$
T_{transit} = \frac{2 R* \sqrt{1 - b^2}}{v}
$$
where $2 R* \sqrt{1 - b^2}$ is the length of the chord that the planet traverses across the stellar disk. The chord length is $2 R*$ if the alignment is perfect ($b=0$). The duration of the transit in time is just this distance divided by the orbital velocity $V$. 

### Ingress / Egress Duration

The planet moves its diameter $2 R_p$ as it moves from first contact to maximally shadowing the disk, and vice versa for egress. The duration is this distance divided by the orbital velocity $v$. The equation is:

$$
t_{ingress} = \frac{2 R_p}{v} = \frac{2 R_p}{2 \pi a/P} = \frac{R_p P}{\pi a}
$$

This shows that we can determine the size of the planet from the ingress / egress time!

### Fitting

We have three observables (transit depth, total duration, and ingress/egress duration) and three unknowns ($R_p/R_*$, $v$, and $b$). Although this assumes we know $R_*$, which we can often infer from models. 

### Simulated transit

In [ ]:
# Physical parameters of the system
rp_rs = 0.10  # Planet-to-star radius ratio (10% = Hot Jupiter)
impact_param = 0.3  # Impact parameter (0 = central, <1 = transiting)
orbital_velocity = 150.0 * u.km / u.s  # Orbital velocity of planet (faster for closer orbit)

# Stellar properties (assume Sun-like star)
R_star_sim = const.R_sun.to(u.km)  # Solar radius

# Calculate observable transit characteristics from physical parameters
depth = rp_rs**2  # Transit depth from geometry

# Transit duration from geometry: T = 2*R_star*sqrt(1-b^2) / v
chord_length = 2 * R_star_sim * np.sqrt(1 - impact_param**2)  # km
transit_duration = (chord_length / orbital_velocity).to(u.day)  # Convert to days

# Ingress/egress duration from geometry: t_ingress = 2*Rp / v
planet_diameter = 2 * rp_rs * R_star_sim  # Diameter of planet in km
ingress_duration = (planet_diameter / orbital_velocity).to(u.day)  # Convert to days

# Observational noise
noise_ppm = 300  # Realistic space-based photometry noise

# Transit center time for simulation (t0 = 0 for convenience)
t0_sim = 0.0

print(f"Physical parameters:")
print(f"  Rp/Rs = {rp_rs:.3f}")
print(f"  Impact parameter b = {impact_param:.2f}")
print(f"  Orbital velocity v = {orbital_velocity.value:.1f} km/s")
print(f"  Stellar radius R* = {R_star_sim.value:.0f} km")
print(f"\nDerived observable characteristics:")
print(f"  Chord length = {chord_length.value:.0f} km (2R* × √(1-b²))")
print(f"  Transit duration = {transit_duration.value:.4f} days = {transit_duration.to(u.hour).value:.2f} hours")
print(f"  Ingress duration = {ingress_duration.value:.6f} days = {ingress_duration.to(u.minute).value:.1f} minutes")
print(f"  Transit depth = {depth:.6f} = {depth*1e6:.0f} ppm")
print(f"\nObservational setup:")
print(f"  Noise level = {noise_ppm} ppm")
print(f"  Signal-to-Noise Ratio (SNR): {depth*1e6/noise_ppm:.1f}")

# Time array: ±3 hours around mid-transit
n_points = 500
times_single = np.linspace(-0.125, 0.125, n_points)  # days

### Here is a trapezoidal model to fit the transit

In [ ]:
def trapezoidal_transit_model(times, t0, depth, duration, ingress_duration):
    """
    Trapezoidal transit model using observable parameters.
    
    Models transit light curve with linear ingress/egress, automatically handling
    both flat-bottomed and grazing (V-shaped) transits based on ingress/duration ratio.
    
    Parameters (all directly observable from light curve):
    - t0: time of transit center [days] (nuisance parameter - observation reference)
    - depth: transit depth (fractional flux decrease, 0-1)
    - duration: total transit duration [days] (Contact I to IV)
    - ingress_duration: duration of ingress or egress [days]
    
    Returns:
    - model_flux: predicted normalized flux values
    
    Note: 
    - If 2*ingress_duration < duration: flat-bottomed transit (trapezoid)
    - If 2*ingress_duration >= duration: grazing transit (triangle/V-shape)
    """
    model_flux = np.ones_like(times)
    
    # Calculate distance from transit center
    dt = times - t0
    abs_dt = np.abs(dt)
    
    # Determine transit shape from observables
    # Flat bottom exists if ingress+egress don't fill entire duration
    flat_duration = duration - 2 * ingress_duration
    has_flat_bottom = (flat_duration > 0)
    
    if has_flat_bottom:
        # STANDARD TRAPEZOIDAL: ingress + flat bottom + egress
        
        # Ingress/egress: linear transitions at edges
        ingress_mask = (abs_dt < duration/2) & (abs_dt >= duration/2 - ingress_duration)
        if np.any(ingress_mask):
            # Fraction of planet on disk increases linearly from 0 to 1
            ingress_frac = (duration/2 - abs_dt[ingress_mask]) / ingress_duration
            model_flux[ingress_mask] = 1 - depth * ingress_frac
        
        # Flat bottom: full depth
        full_mask = abs_dt < (duration/2 - ingress_duration)
        model_flux[full_mask] = 1 - depth
        
    else:
        # GRAZING TRANSIT: V-shaped (ingress blends directly into egress)
        # Depth varies linearly from maximum at center to zero at edges
        
        transit_mask = abs_dt < duration/2
        if np.any(transit_mask):
            # Fraction from center (0 at center, 1 at edge)
            frac_from_center = abs_dt[transit_mask] / (duration/2)
            # Depth decreases linearly from center to edge
            model_flux[transit_mask] = 1 - depth * (1 - frac_from_center)
    
    return model_flux

# True transit signal (no noise) - using the trapezoidal model with observable parameters
flux_true = trapezoidal_transit_model(times_single, t0_sim, depth, transit_duration.value, ingress_duration.value)

# Add Gaussian noise
noise_per_point = noise_ppm * 1e-6
noise = np.random.normal(0, noise_per_point, n_points)
flux_observed = flux_true + noise
flux_err = np.full(n_points, noise_per_point)

print(f"\nSignal-to-noise ratio: {depth/noise_per_point:.1f}")

# Plot the single transit observation
plt.figure(figsize=(10, 6))
plt.errorbar(times_single * 24, flux_observed, yerr=flux_err,
             fmt='o', markersize=4, alpha=0.6, color='black',
             ecolor='gray', capsize=2, label='Simulated observations')
plt.plot(times_single * 24, flux_true, 'r-', linewidth=2.5,
         alpha=0.7, label='True model')
plt.xlabel('Time from Transit Center [hours]', fontsize=14)
plt.ylabel('Relative Flux', fontsize=14)
plt.title('Simulated Single Transit Observation', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.axhline(1.0, color='k', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()

### Fitting with `scipy.optimize.curve_fit`

In [ ]:
# Set bounds to guide the optimizer and prevent unphysical solutions
# Parameters: [t0, depth, duration, ingress_duration]
# For ingress: realistic range is T_total * (Rp/Rs) / sqrt(1-b²)
#   For Rp/Rs up to 0.15 and b<0.9: ingress < 0.015 days
bounds = (
    [-0.05, 0.0001, 0.05, 0.001],      # Lower bounds
    [ 0.05, 0.05, 0.25, 0.015]         # Upper bounds: tighter ingress bound for stability
)

# Multiple Random Starts Strategy
# Try many different initial guesses and pick the best result
n_tries = 30  # Number of random starting points to try
results = []

print(f"Trying {n_tries} random initial guesses...")

for i in range(n_tries):
    # Generate random initial guesses within reasonable ranges
    # Parameters: [t0, depth, duration, ingress_duration]
    random_t0 = np.random.uniform(-0.04, 0.04)  # Mid-transit time around 0
    random_depth = np.random.uniform(0.0005, 0.01)  # Transit depth (0.05% to 1%)
    random_duration = np.random.uniform(0.08, 0.22)  # Total duration in days
    random_ingress = np.random.uniform(0.002, 0.012)  # Ingress duration in days
    
    p0_random = [random_t0, random_depth, random_duration, random_ingress]
    
    try:
        # Try fitting with this random initial guess
        popt_try, pcov_try = optimize.curve_fit(
            trapezoidal_transit_model,
            times_single,
            flux_observed,
            p0=p0_random,
            sigma=flux_err,
            absolute_sigma=True,
            bounds=bounds,
            maxfev=10000  # Allow more iterations for difficult fits
        )
        
        # Calculate chi-squared for this fit
        model_try = trapezoidal_transit_model(times_single, *popt_try)
        chi_sq = np.sum(((flux_observed - model_try) / flux_err)**2)
        
        results.append({
            'chi_squared': chi_sq,
            'fitted_params': popt_try,
            'covariance': pcov_try,
            'success': True
        })
    
    except Exception as e:
        results.append({'success': False})

# Find the best result (lowest chi-squared)
successful_results = [r for r in results if r.get('success', False)]

if len(successful_results) > 0:
    best_result = min(successful_results, key=lambda x: x['chi_squared'])
    chi_sq_values = [r['chi_squared'] for r in successful_results]
    
    # Extract best fit parameters (observables)
    popt = best_result['fitted_params']
    pcov = best_result['covariance']
    t0_fit, depth_fit, duration_fit, ingress_fit = popt
    errors = np.sqrt(np.diag(pcov))
    
    print(f"\n{len(successful_results)}/{n_tries} fits succeeded")
    print(f"χ² range: [{min(chi_sq_values):.1f}, {max(chi_sq_values):.1f}]")
    print(f"Best χ² = {best_result['chi_squared']:.1f}")
    
    print("\nFitted Observable Parameters (Best of 30 attempts):")
    print("="*60)
    print(f"t0           = {t0_fit:10.6f} ± {errors[0]:.6f} days")
    print(f"Depth        = {depth_fit:10.6f} ± {errors[1]:.6f} ({depth_fit*1e6:.0f} ± {errors[1]*1e6:.0f} ppm)")
    print(f"Duration     = {duration_fit:10.6f} ± {errors[2]:.6f} days ({duration_fit*24:.2f} ± {errors[2]*24:.2f} hours)")
    print(f"Ingress dur  = {ingress_fit:10.6f} ± {errors[3]:.6f} days ({ingress_fit*24*60:.1f} ± {errors[3]*24*60:.1f} min)")
else:
    print(f"\nWARNING: No fits succeeded out of {n_tries} attempts!")
    print("There may be an issue with the bounds or initial guesses.")

### Calculate the fit and visualize the result

In [ ]:
# Generate model with fitted parameters
model_fit = trapezoidal_transit_model(times_single, *popt)

# Calculate chi-squared
chi2 = np.sum(((flux_observed - model_fit) / flux_err)**2)
dof = len(flux_observed) - len(popt)
chi2_red = chi2 / dof

print(f"\nFit quality:")
print(f"  χ² = {chi2:.1f}")
print(f"  DOF = {dof}")
print(f"  Reduced χ² = {chi2_red:.3f}")

# For visualization, show one example initial guess (not used in actual fitting)
# Parameters: [t0, depth, duration, ingress_duration]
p0_example = [0.0, 0.002, 0.13, 0.015]  # Example guess: slightly wrong depth and durations
model_example = trapezoidal_transit_model(times_single, *p0_example)

# Plot: data, example initial guess, and best fit from multiple random starts
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

# Top panel: Full transit
ax1.errorbar(times_single * 24, flux_observed, yerr=flux_err,
             fmt='o', markersize=4, alpha=0.5, color='black',
             ecolor='lightgray', capsize=2, label='Observed data')
ax1.plot(times_single * 24, model_example, 'r--', linewidth=2,
         alpha=0.6, label='Example single guess')
ax1.plot(times_single * 24, model_fit, 'b-', linewidth=2.5,
         label=f'Best fit (30 random starts, χ²_red={chi2_red:.3f})')
ax1.set_ylabel('Relative Flux', fontsize=14)
ax1.set_title('Transit Fit: Multiple Random Starts Find Optimal Solution', fontsize=15, fontweight='bold')
ax1.legend(fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-3, 3)

# Bottom panel: Zoom on ingress
ingress_mask = (times_single * 24 > -2.5) & (times_single * 24 < -1)
ax2.errorbar(times_single[ingress_mask] * 24, flux_observed[ingress_mask],
             yerr=flux_err[ingress_mask],
             fmt='o', markersize=6, alpha=0.7, color='black',
             ecolor='gray', capsize=2, label='Data')
ax2.plot(times_single[ingress_mask] * 24, model_example[ingress_mask],
         'r--', linewidth=2.5, alpha=0.7, label='Example single guess')
ax2.plot(times_single[ingress_mask] * 24, model_fit[ingress_mask],
         'b-', linewidth=3, label='Best fit')
ax2.set_xlabel('Time from Transit Center [hours]', fontsize=14)
ax2.set_ylabel('Relative Flux', fontsize=14)
ax2.set_title('Ingress Detail: Fit Quality', fontsize=14)
ax2.legend(fontsize=12)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Compare to the input

In [ ]:
print("\nObservable Parameter Recovery:")
print("="*70)
print(f"{'Observable':<20} {'True':<12} {'Fitted':<15} {'Error (sigma)':<15}")
print("-"*70)

# True values of fitted observable parameters
true_vals = [t0_sim, depth, transit_duration.value, ingress_duration.value]
param_names = ['t0 [days]', 'Depth', 'Duration [days]', 'Ingress [days]']

for i, (name, true, fitted, err) in enumerate(zip(param_names, true_vals, popt, errors)):
    sigma_off = abs(true - fitted) / err if err > 0 else 0
    print(f"{name:<20} {true:<12.6f} {fitted:<15.6f} {sigma_off:<15.2f}")

# Now convert fitted observables back to physical parameters
print("\n" + "="*70)
print("Converting Fitted Observables → Physical Parameters:")
print("="*70)

# 1. Planet-to-star radius ratio from depth
rp_rs_recovered = np.sqrt(depth_fit)

# 2. Impact parameter and velocity from duration and ingress
# From geometry:
#   T_total = 2*R_star*sqrt(1-b^2) / v
#   T_ingress = 2*Rp / v = 2*(Rp/Rs)*R_star / v
# Taking ratio: T_ingress/T_total = (Rp/Rs) / sqrt(1-b^2)
# Solving: sqrt(1-b^2) = (Rp/Rs) * T_total / T_ingress

sqrt_1_minus_b2 = rp_rs_recovered * duration_fit / ingress_fit
# Clamp to valid range to avoid numerical issues
sqrt_1_minus_b2 = min(sqrt_1_minus_b2, 0.9999)  # b must be < 1
b_recovered = np.sqrt(1 - sqrt_1_minus_b2**2)

# 3. Orbital velocity from total duration and geometry
chord_recovered = 2 * R_star_sim * sqrt_1_minus_b2
duration_recovered = duration_fit * u.day
v_recovered = (chord_recovered / duration_recovered).to(u.km / u.s)

print(f"\n1. Planet-to-star radius ratio:")
print(f"   Rp/Rs = sqrt(depth) = sqrt({depth_fit:.6f}) = {rp_rs_recovered:.4f}")
print(f"   True value: {rp_rs:.4f}")
print(f"   Recovery error: {abs(rp_rs_recovered - rp_rs)/rp_rs * 100:.2f}%")

print(f"\n2. Impact parameter:")
print(f"   From T_ingress/T_total ratio: {ingress_fit/duration_fit:.4f}")
print(f"   sqrt(1-b²) = {sqrt_1_minus_b2:.4f}")
print(f"   b = {b_recovered:.3f}")
print(f"   True value: {impact_param:.3f}")
print(f"   Recovery error: {abs(b_recovered - impact_param):.3f}")

print(f"\n3. Orbital velocity:")
print(f"   v = 2*R*×sqrt(1-b²) / T_total")
print(f"   v = {v_recovered.value:.2f} km/s")
print(f"   True value: {orbital_velocity.value:.2f} km/s")
print(f"   Recovery error: {abs(v_recovered - orbital_velocity).value:.2f} km/s ({abs(v_recovered - orbital_velocity).value/orbital_velocity.value * 100:.2f}%)")

# 4. Planet radius in physical units (assuming Sun-like star)
print(f"\n4. Planet physical radius:")
R_star = const.R_sun.to(u.km)
R_planet = rp_rs_recovered * R_star
R_planet_earth = (R_planet / const.R_earth.to(u.km)).value
R_planet_jupiter = (R_planet / const.R_jup.to(u.km)).value
print(f"   Assuming R* = R☉ = {R_star.value:.0f} km")
print(f"   R_planet = {R_planet.value:.0f} km")
print(f"            = {R_planet_earth:.2f} R_Earth")
print(f"            = {R_planet_jupiter:.2f} R_Jupiter")
print(f"   (Hot Jupiter)")

## Section 6: Observing multiple transits

The period of the orbit is a critical parameter. It is necessary to measure the size of the planet and to determine the orbital velocity. 

Exoplanet discovery requires detection of at least two transits. Multiple transits help with many things:
1. Confirm that there is a transit
2. Confirm that it is periodic
3. Provide multiple transits to obtain a better fit

Now we'll generate a multi-transit light curve.

In [ ]:
# Same physical parameters as before - reuse from single transit
# rp_rs, impact_param, orbital_velocity all remain the same
# transit_duration was calculated from these physical parameters

# Add orbital period for multi-transit scenario
period = 2.55  # days - short orbital period

print(f"Generating multi-transit light curve:")
print(f"  Period = {period:.2f} days")
print(f"  Physical parameters (same as single transit):")
print(f"    Rp/Rs = {rp_rs:.3f}")
print(f"    Impact parameter b = {impact_param:.2f}")
print(f"    Orbital velocity v = {orbital_velocity.value:.1f} km/s")
print(f"  Calculated transit duration = {transit_duration.value:.4f} days = {(transit_duration*24).value:.1f} hours")
print(f"  Observing span = 30 days")
print(f"  Expected number of transits = {30/period:.0f}")

def multi_transit_model(times, t0, period, depth, duration, ingress_duration):
    """
    Model for multiple transits with given period.
    Uses trapezoidal model with observable parameters for each transit.
    
    Parameters:
    - t0: time of first transit center
    - period: orbital period
    - depth, duration, ingress_duration: observable transit parameters
    
    Generates transit dips at t0, t0+P, t0+2P, ...
    """
    flux = np.ones_like(times)
    
    # For each time point, find closest transit center
    # and calculate that transit's contribution
    for t in times:
        # Phase within period
        phase = ((t - t0) % period) - period/2
        if abs(phase) < period/2:
            phase = phase if abs(phase) < abs(phase + period) else phase + period
        
        # Generate transit if within duration
        if abs(phase) < duration:
            flux_at_t = trapezoidal_transit_model(np.array([phase]), 0.0, depth, duration, ingress_duration)[0]
            idx = np.where(times == t)[0]
            if len(idx) > 0:
                flux[idx] = flux_at_t
    
    return flux

#########################################
# Generate the multi-transit time series
#########################################

# Use same realistic noise level as single transit
noise_per_point_multi = noise_ppm * 1e-6

print(f"\n{'='*60}")
print(f"MULTI-TRANSIT SCENARIO")
print(f"{'='*60}")
print(f"Transit depth: {depth*1e6:.0f} ppm")
print(f"Noise level: {noise_ppm} ppm")
print(f"Single-transit SNR: {depth/noise_per_point_multi:.1f} (excellent)")
print(f"After phase-folding 12 transits: SNR ~ {(depth/noise_per_point_multi)*np.sqrt(12):.1f} (extremely precise)")

# Time array: 30 days with 10-minute cadence
time_span = 30.0  # days
cadence = 10.0 / (24 * 60)  # 10 minutes in days
times_multi = np.arange(0, time_span, cadence)

# Generate clean multi-transit signal (using trapezoidal model with observable parameters)
flux_multi_true = multi_transit_model(times_multi, 0.0, period, depth, transit_duration.value, ingress_duration.value)

# Add realistic higher noise
noise_multi = np.random.normal(0, noise_per_point_multi, len(times_multi))
flux_multi_observed = flux_multi_true + noise_multi
flux_multi_err = np.full(len(times_multi), noise_per_point_multi)

print(f"\nGenerated {len(times_multi)} data points over {time_span} days")
print(f"Number of transit events: {np.sum(flux_multi_true < 0.999)//50:.0f}")

In [ ]:
# Plot the full 30-day light curve
plt.figure(figsize=(14, 5))
plt.plot(times_multi, flux_multi_observed, 'k.', markersize=2, alpha=0.5, label='Observations')
plt.plot(times_multi, flux_multi_true, 'r-', linewidth=1, alpha=0.7, label='True model')
plt.xlabel('Time [days]', fontsize=14)
plt.ylabel('Relative Flux', fontsize=14)
plt.title(f'Multi-Transit Light Curve: Hot Jupiter (P = {period} days)', fontsize=15, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.xlim(0, time_span)
plt.ylim(0.985, 1.002)  # Wider range to show transits and baseline clearly
plt.tight_layout()
plt.show()

## Solutions to In-Class Exercises

### Section 1 Solution

In [ ]:
planets = [
    ("Hot Jupiter (3d)", M_jupiter, 3.0),
    ("Warm Neptune (20d)", 17*M_earth, 20.0),
    ("Super-Earth (50d)", 5*M_earth, 50.0),
    ("Earth twin (365d)", M_earth, 365.25),
    ("Solar System Jupiter (11.86y)", M_jupiter, 11.86*365.25),
]

print("\nRadial Velocity Amplitudes for Different Planets:")
print("-" * 60)
print(f"{'Planet Type':<30} {'Mass':<15} {'Period':<10} {'K (m/s)':<10}")
print("-" * 60)

for name, mass, period_days in planets:
    period_s = (period_days * u.day).to(u.s).value
    
    # Calculate RV amplitude
    K = (2 * np.pi * G / period_s)**(1/3) * (mass * sin_i) / (m_star)**(2/3)
    
    # Mass in Earth masses for display
    mass_earth = mass / M_earth
    
    print(f"{name:<30} {mass_earth:>6.1f} M⊕  {period_days:>8.1f}d  {K:>8.2f}")

### Section 3 Solution

In [ ]:
P_transit_jupiter = (R_star / a_jupiter).decompose()

print(f"Jupiter at {a_jupiter}:")
print(f"  P_transit ≈ {R_star:.5f} / {a_jupiter}")
print(f"  P_transit ≈ {P_transit_jupiter:.5f} = {P_transit_jupiter*100:.3f}%")
print()
print(f"Ratio: Jupiter is {P_transit_jupiter/P_transit_earth:.1f}× more likely to transit")